# Options Scanner — v4.27
**CALL · PUT · CSP signals with 11-point quality scoring gate**

Aligned with Jason Brown's PTU trading framework.
Backtest validated: 50.1% win rate, 2.06 profit factor (34 symbols, 2023–2024).

---

### Signal Types
| Signal | Platform | Regime Required |
|--------|----------|-----------------|
| BUY CALL | Robinhood or thinkorswim | BULLISH only |
| BUY PUT | Robinhood or thinkorswim | Any (9+ score in BULLISH) |
| SELL PUT (CSP) | thinkorswim only | BULLISH or MIXED |

### Scoring (11 pts total)
Regime (2) + RSI (2) + Trend (2) + Volume (2) + Weekly (1) + Support (1) + Bollinger (1)
MACD is informational only — not scored (backtest showed not predictive)

### Conviction
| Score | Conviction | Action |
|-------|------------|--------|
| 10–11 | VERY HIGH | Strong candidate |
| 8–9 | HIGH | Good candidate |
| 7 | MODERATE | Minimum threshold |
| < 7 | NONE | No signal — wait |

### Phase 3 — Regime Posture
| Regime | CALLs | PUTs | CSPs |
|--------|-------|------|------|
| BULLISH | ✅ Score 7+ | ✅ Score 9+ only | ✅ |
| MIXED stable | ❌ Score 9+ required | ✅ | ✅ |
| MIXED deteriorating | ❌ Blocked | ✅ | ✅ |
| MIXED recovering | ❌ Wait for BULLISH | ✅ | ✅ |
| BEARISH | ❌ Blocked | ✅ | ❌ |

### Stop Orders in Robinhood
⚠️ Always use **Stop Limit** — NOT limit sell.
Limit sell fills immediately at open if bid exceeds your price.

### Daily Workflow
1. Run **Force Reload Cell**
2. Run **Cell 3** — set account size
3. Run **Cell 4** — run the scan
4. If trading: run **Cell 7** with fill price


In [2]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy'])
print('Dependencies ready')

Dependencies ready


In [3]:
# ─────────────────────────────────────────────────────────────
# FORCE RELOAD CELL — run this whenever options_scanner_v4.py
# is updated on GitHub. DO NOT DELETE.
# ─────────────────────────────────────────────────────────────
import sys, os, time

if 'options_scanner_v4' in sys.modules:
    del sys.modules['options_scanner_v4']
if os.path.exists('options_scanner_v4.py'):
    os.remove('options_scanner_v4.py')

url = f"https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py?t={int(time.time())}"
!wget -q -O options_scanner_v4.py "{url}"

# Verify correct version loaded
with open('options_scanner_v4.py') as f:
    content = f.read()

checks = ['target_width', 'min_reward_risk', '2.0', 'also_qualified', 'hard_fail']
for c in checks:
    print(f"{'✓' if c in content else '✗'} {c}")

from options_scanner_v4 import run_scan, print_results, deep_dive, ALL_SYMBOLS, CONFIG, find_best_call
print('\n✓ Scanner reloaded successfully')

✓ target_width
✓ min_reward_risk
✓ 2.0
✓ also_qualified
✓ hard_fail

✓ Scanner reloaded successfully


In [4]:
# Cell 2 — Load scanner from GitHub
# Replace YOUR_USERNAME with kevin-mumaw
!wget -q https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py

from options_scanner_v4 import (
    run_scan, print_results, deep_dive,
    ALL_SYMBOLS, WATCHLIST, CONFIG
)
print('Scanner loaded')
print(f'Watchlist: {len(ALL_SYMBOLS)} symbols')
print('Groups:', list(WATCHLIST.keys()))

Scanner loaded
Watchlist: 34 symbols
Groups: ['tier1_affordable', 'tier2_sweet_spot', 'tier3_mid_price', 'tier4_high_price']


In [5]:
# Cell 3 — Configure
# Edit account size here. Everything else auto-adjusts.

MY_CONFIG = {**CONFIG}  # copy defaults

MY_CONFIG['account_size'] = 5000.00  # update as your account grows

# Optional: scan only specific symbols instead of full watchlist
# MY_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM']
MY_SYMBOLS = None  # None = scan all 25 symbols

print(f'Account: ${MY_CONFIG["account_size"]:,.0f}')
print(f'Min score to generate signal: {MY_CONFIG["min_score"]}/10')
print(f'Stop loss: -{int(MY_CONFIG["stop_loss_pct"]*100)}% | '
      f'Profit target: +{int(MY_CONFIG["profit_target_pct"]*100)}% | '
      f'Time stop: {MY_CONFIG["time_stop_dte"]} DTE')

Account: $5,000
Min score to generate signal: 7/10
Stop loss: -35% | Profit target: +75% | Time stop: 21 DTE


In [6]:
# Cell 4 — Run the scan
# This is the main cell. Run this daily.
results = run_scan(symbols=MY_SYMBOLS, config=MY_CONFIG)
print_results(results)


  Fetching market regime (QQQ)...
  Regime: BULLISH — QQQ 738.31 | MA50 652.9 (+13.1%) | MA200 616.82 (+19.7%)

  Scanning 34 symbols.................................. done.


██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — v4.27  |  2026-05-31 18:26 ET
  Account: $5,000  |  Scanned: 34 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 738.31 | MA50 652.9 (+13.1%) | MA200 616.82 (+19.7%)

  📋 Market extended above MA50 — proceed only with highest conviction setups

  4 signal(s) found:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SIGNAL #1  |  BAC — BUY CALL  |  Score: 8/10  [HIGH]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  WHY THIS TRADE:
    ✓ Regime [2/2] BULLISH — QQQ above both MAs
    ~ RSI     [1/2] Neutral — RSI 52 (room to run)
    ✓ Trend   [2/2] Clean uptrend — price > MA20 > MA50
    ✓ Weekly  [1/1] Weekly uptrend confirmed
    ~ Volume  [1/

In [ ]:
# Cell 5 — Deep dive on a specific symbol
# Use this to investigate any symbol in detail,
# whether it appeared in the scan or not.

SYMBOL = 'SOFI'  # change this

deep_dive(SYMBOL, config=MY_CONFIG)


  Fetching regime and analyzing SOFI...

██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  v4.22  |  2026-05-31 00:42
  Account: $5,000  |  Scanned: 1 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 738.31 | MA50 652.9 (+13.1%) | MA200 616.82 (+19.7%)

  📋 Market extended above MA50 — proceed only with highest conviction setups

──────────────────────────────────────────────────────────────
  NO QUALIFYING SETUPS TODAY
  Recommendation: Wait. Do not force a trade.

  Best scores today:
    SOFI    5/10  Trend: MIXED       RSI: 69.6

──────────────────────────────────────────────────────────────



In [ ]:
# Cell 6 — Review all scores from last scan
# See every symbol's score ranked highest to lowest.

import pandas as pd

rows = []
for r in results['all_results']:
    if not r.get('error'):
        rows.append({
            'Symbol':    r['symbol'],
            'Score':     r['score'],
            'Conviction':r['conviction'],
            'Trend':     r.get('trend','—'),
            'RSI':       r.get('rsi','—'),
            'Volume':    r.get('vol_data',{}).get('label','—'),
            'Weekly':    r.get('weekly',{}).get('trend','—'),
            'Has Option':r.get('option') is not None,
        })

df = pd.DataFrame(rows).sort_values('Score', ascending=False)
print(df.to_string(index=False))

Symbol  Score Conviction     Trend  RSI            Volume    Weekly  Has Option
   BAC      8       HIGH   UPTREND 52.2           NEUTRAL   UPTREND        True
     C      8       HIGH   UPTREND 50.9           NEUTRAL   UPTREND       False
   XLF      7   MODERATE   UPTREND 53.7           NEUTRAL     MIXED        True
    KO      7   MODERATE     MIXED 53.2           NEUTRAL   UPTREND        True
   DIS      7   MODERATE     MIXED 33.2           NEUTRAL     MIXED       False
  UBER      7   MODERATE     MIXED 25.5           NEUTRAL DOWNTREND        True
 GOOGL      7   MODERATE     MIXED 35.0           NEUTRAL   UPTREND       False
  AMZN      7   MODERATE   UPTREND 47.9           NEUTRAL   UPTREND       False
   AAL      6   MODERATE   UPTREND 64.1      ACCUMULATION     MIXED       False
   AMD      6   MODERATE   UPTREND 67.0           NEUTRAL   UPTREND       False
  INTC      6   MODERATE     MIXED 39.9           NEUTRAL   UPTREND       False
    MU      6   MODERATE   UPTREND 70.1 

In [ ]:
# Cell 7 — Trade Journal (entry, exit, and re-entry)
import os
if os.path.exists("trade_journal.csv"):
    os.remove("trade_journal.csv")
    print("Journal cleared")

# Trade 1 — Original buy 5/26
trade_id_1 = log_entry(
    symbol         = "BAC",
    direction      = "BUY CALL",
    trade_type     = "SINGLE LEG",
    score          = 9,
    conviction     = "HIGH",
    market_regime  = "BULLISH",
    entry_premium  = 3.45,
    stop_price     = 2.24,
    target_price   = 6.04,
    time_stop_date = "2026-06-26",
    contracts      = 1,
    total_cost     = 345,
    notes          = "First live trade — entered 5/26"
)

log_exit(
    trade_id     = trade_id_1,
    exit_premium = 3.55,
    exit_reason  = "MANUAL",
    notes        = "Accidental exit 5/27 at open — limit sell error. Should have been stop limit."
)

# Trade 2 — Re-entry 5/27
trade_id_2 = log_entry(
    symbol         = "BAC",
    direction      = "BUY CALL",
    trade_type     = "SINGLE LEG",
    score          = 9,
    conviction     = "HIGH",
    market_regime  = "BULLISH",
    entry_premium  = 3.45,
    stop_price     = 2.24,
    target_price   = 6.04,
    time_stop_date = "2026-06-26",
    contracts      = 1,
    total_cost     = 345,
    notes          = "Re-entry 5/27 after accidental exit"
)

print(f"\nOpen trade ID: {trade_id_2}")
show_journal()

NameError: name 'log_entry' is not defined

In [ ]:
# Cell 9 — View journal anytime
show_journal()
# show_journal("OPEN")
# show_journal("WIN")

NameError: name 'show_journal' is not defined

In [ ]:
# Cell 10 — Backtest
from options_scanner_v4 import run_backtest, print_backtest_results, export_backtest

df = run_backtest(
    symbols    = ALL_SYMBOLS,
    start_date = "2020-01-01",
    end_date   = "2024-12-31",
    min_score  = 7,   # matches scanner CONFIG
    stop_pct   = 0.35,
)
print_backtest_results(df)


  BACKTEST — v4 Scoring System
  Period: 2020-01-01 to 2024-12-31
  Symbols: 34  |  Min score: 7/12
  Hold: 17d  |  Stop: -35%  |  Target: +75%

  Fetching QQQ regime data...
  BAC... 34 trades
  F... 28 trades
  PLTR... 8 trades
  T... 23 trades
  PFE... 24 trades
  AAL... 26 trades
  SOFI... 9 trades
  XLF... 32 trades
  KO... 32 trades
  DIS... 25 trades
  NKE... 28 trades
  UBER... 22 trades
  AMD... 31 trades
  INTC... 22 trades
  WFC... 26 trades
  C... 33 trades
  MU... 28 trades
  AAPL... 25 trades
  GOOGL... 27 trades
  JPM... 32 trades
  V... 28 trades
  MA... 29 trades
  XOM... 28 trades
  CVX... 28 trades
  UNH... 29 trades
  JNJ... 31 trades
  GS... 35 trades
  MSFT... 33 trades
  AMZN... 25 trades
  META... 36 trades
  NVDA... 27 trades
  COST... 27 trades
  SPY... 25 trades
  QQQ... 26 trades

  BACKTEST RESULTS — v4 Scoring System
  Total trades   : 922
  Win rate       : 42.7%  (394W / 528L)
  Total P&L      : $+48,307.34
  Avg win        : $+340.75
  Avg loss       :

In [ ]:
import requests
url = "https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py"
r = requests.get(url)
print(r.status_code)
print(r.text[:200])


200
"""
options_scanner_v4.py
Phase 1 — CALL-side scanner with quality scoring gate.
Aligned with Jason Brown's PTU trading principles.

Design philosophy:
  - Only generate a signal


In [ ]:
import sys, os, importlib

# Nuclear clear
for mod in list(sys.modules.keys()):
    if 'options_scanner' in mod:
        del sys.modules[mod]

if os.path.exists('options_scanner_v4.py'):
    os.remove('options_scanner_v4.py')
    print("Deleted old file")

import requests
url = "https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py"
r = requests.get(url)
with open('options_scanner_v4.py', 'w') as f:
    f.write(r.text)
print(f"Downloaded {len(r.text)} bytes")

# Verify version
import options_scanner_v4 as s
print(f"VERSION: {s.VERSION}")

Deleted old file
Downloaded 112461 bytes
VERSION: 4.23


In [ ]:
import options_scanner_v4 as s
print(s.VERSION)


4.26
